# Encoder Metrics Comparison

Run this notebook only while attached to the encoder Docker container (attached mode).

Comparison of `ALIGN`, `CLIP`, `BLIP`, and `SIGLIP` embedders on nuImages.

What this notebook does:
1. Computes retrieval metrics: `Recall@K`, `Precision@K`, `nDCG@K`.
2. Compares models on charts (side-by-side subplots per model).
3. Benchmarks inference speed (image/text) and plots it.


In [1]:
# Install notebook dependencies
!pip install -qU matplotlib ipywidgets jupyterlab_widgets

In [2]:
from __future__ import annotations

import gc
import json
import time
from collections import Counter, defaultdict
from pathlib import Path
from typing import Dict, List

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from tqdm.auto import tqdm
from transformers import (
    AlignModel,
    AlignProcessor,
    BlipModel,
    BlipProcessor,
    CLIPModel,
    CLIPProcessor,
    SiglipModel,
    SiglipProcessor,
)

# Switch to 'cuda' if a GPU is available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# Global experiment settings
ROOT = "../data/nuimages"
META = f"{ROOT}/v1.0-train"
LIMIT = 2000                 # max number of images used for evaluation
TOP_K_CLASSES = 50           # number of most frequent classes to compare
K_LIST = [1, 5, 10, 20, 50]

# Batch sizes for faster inference
IMAGE_BATCH_SIZE = 16
TEXT_BATCH_SIZE = 32


Device: cpu


## 1) Data Preparation


In [3]:
# Check that data is available
!ls ../data/nuimages/
# Uncomment if you need to unpack the archive:
# !tar -xzf ../data/nuimages/nuimages-v1.0-all-samples.tgz -C ../data/nuimages/


LICENSE				samples    v1.0-train
nuimages-v1.0-all-metadata.tgz	v1.0-mini  v1.0-val
nuimages-v1.0-all-samples.tgz	v1.0-test


In [4]:
# Load nuImages and build dataset: image -> set(classes)
object_ann = json.load(open(f"{META}/object_ann.json"))
category = {c["token"]: c["name"] for c in json.load(open(f"{META}/category.json"))}
sample_data = {s["token"]: s for s in json.load(open(f"{META}/sample_data.json"))}


def norm_cat(cat: str) -> str:
    # vehicle.car -> car
    return cat.split(".")[-1]


image_to_objs = defaultdict(list)
for ann in object_ann:
    sd_token = ann["sample_data_token"]
    cat = norm_cat(category[ann["category_token"]])
    image_to_objs[sd_token].append(cat)


dataset = []
for sd_token, cats in image_to_objs.items():
    path = sample_data[sd_token]["filename"]
    dataset.append({
        "image_path": f"{ROOT}/{path}",
        "classes": set(cats),
    })

print(f"Dataset size: {len(dataset)}")

# Class frequency distribution
counter = Counter()
for item in dataset:
    for c in item["classes"]:
        counter[c] += 1

CLASSES = [c for c, _ in counter.most_common(TOP_K_CLASSES)]
print("Top classes:", CLASSES[:15])


Dataset size: 60668
Top classes: ['car', 'adult', 'truck', 'trafficcone', 'barrier', 'motorcycle', 'bicycle', 'rigid', 'construction_worker', 'construction', 'bicycle_rack', 'pushable_pullable', 'trailer', 'debris', 'child']


## 2) Model Setup (ALIGN / CLIP / BLIP / SIGLIP)

Unified interface for all embedders so metrics and speed are computed consistently.


In [5]:
MODEL_SPECS = {
    "align": {
        "label": "ALIGN",
        "model_id": "kakaobrain/align-base",
        "processor_cls": AlignProcessor,
        "model_cls": AlignModel,
    },
    "clip": {
        "label": "CLIP",
        "model_id": "openai/clip-vit-base-patch32",
        "processor_cls": CLIPProcessor,
        "model_cls": CLIPModel,
    },
    "blip": {
        "label": "BLIP",
        "model_id": "Salesforce/blip-image-captioning-base",
        "processor_cls": BlipProcessor,
        "model_cls": BlipModel,
    },
    "siglip": {
        "label": "SIGLIP",
        "model_id": "google/siglip-base-patch16-224",
        "processor_cls": SiglipProcessor,
        "model_cls": SiglipModel,
    },
}

_loaded_models: Dict[str, dict] = {}


def load_embedder(model_key: str) -> dict:
    if model_key in _loaded_models:
        return _loaded_models[model_key]

    spec = MODEL_SPECS[model_key]
    print(f"Loading {spec['label']} from {spec['model_id']}...")

    processor = spec["processor_cls"].from_pretrained(spec["model_id"])
    model = spec["model_cls"].from_pretrained(spec["model_id"]).to(device)
    model.eval()

    pack = {
        "label": spec["label"],
        "model_id": spec["model_id"],
        "processor": processor,
        "model": model,
    }
    _loaded_models[model_key] = pack
    return pack


def _normalize_torch(x: torch.Tensor) -> torch.Tensor:
    return x / (x.norm(dim=-1, keepdim=True) + 1e-12)


def _extract_features(outputs) -> torch.Tensor:
    # In newer HF versions, get_*_features often already returns a Tensor,
    # but we keep this safe fallback.
    if isinstance(outputs, tuple):
        outputs = outputs[0]
    if hasattr(outputs, "pooler_output"):
        outputs = outputs.pooler_output
    if outputs.ndim == 3:
        outputs = outputs[:, 0, :]
    return outputs


def encode_texts(model_key: str, texts: List[str], batch_size: int = TEXT_BATCH_SIZE) -> np.ndarray:
    pack = load_embedder(model_key)
    processor, model = pack["processor"], pack["model"]

    all_embs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]

        text_kwargs = {"text": batch, "return_tensors": "pt", "truncation": True}
        if model_key == "siglip":
            # Recommended for SigLIP in the docs
            text_kwargs["padding"] = "max_length"
        else:
            text_kwargs["padding"] = True

        inputs = processor(**text_kwargs)
        inputs = {k: v.to(device) for k, v in inputs.items() if k in {"input_ids", "attention_mask", "token_type_ids", "position_ids"}}

        with torch.inference_mode():
            outputs = model.get_text_features(**inputs)
            outputs = _extract_features(outputs)
            outputs = _normalize_torch(outputs)

        all_embs.append(outputs.cpu().numpy())

    return np.vstack(all_embs)


def encode_images_from_paths(model_key: str, image_paths: List[str], batch_size: int = IMAGE_BATCH_SIZE) -> np.ndarray:
    pack = load_embedder(model_key)
    processor, model = pack["processor"], pack["model"]

    all_embs = []
    for i in tqdm(range(0, len(image_paths), batch_size), desc=f"Images [{pack['label']}]"):
        batch_paths = image_paths[i:i + batch_size]
        images = []
        for p in batch_paths:
            with Image.open(p) as img:
                images.append(img.convert("RGB"))

        inputs = processor(images=images, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items() if k == "pixel_values"}

        with torch.inference_mode():
            outputs = model.get_image_features(**inputs)
            outputs = _extract_features(outputs)
            outputs = _normalize_torch(outputs)

        all_embs.append(outputs.cpu().numpy())

    return np.vstack(all_embs)


def clear_model_cache():
    if device == "cuda":
        torch.cuda.empty_cache()
    gc.collect()


## 3) Metrics Evaluation


In [ ]:
def evaluate_class(sim: np.ndarray, labels: np.ndarray, k: int):
    order = np.argsort(-sim)
    top_k = order[:k]

    positives = labels.sum()
    recall = labels[top_k].sum() / positives if positives > 0 else 0.0
    precision = labels[top_k].sum() / k

    rel = labels[order[:k]]
    dcg = sum(r / np.log2(i + 2) for i, r in enumerate(rel))
    idcg = sum(1 / np.log2(i + 2) for i in range(min(k, int(positives))))
    ndcg = dcg / idcg if idcg > 0 else 0.0

    return float(recall), float(precision), float(ndcg)


def evaluate_single_model(model_key: str, dataset_subset: list, classes: list, k_list: list):
    model_label = MODEL_SPECS[model_key]["label"]
    print()
    print(f"===== Evaluating {model_label} =====")

    image_paths = [item["image_path"] for item in dataset_subset]
    image_embs = encode_images_from_paths(model_key, image_paths)

    prompts = [f"photo of a {cls}" for cls in classes]
    text_embs = encode_texts(model_key, prompts)

    sim_matrix = image_embs @ text_embs.T

    # labels_by_class[cls] = binary vector over all images
    labels_by_class = {
        cls: np.array([1 if cls in item["classes"] else 0 for item in dataset_subset], dtype=np.int32)
        for cls in classes
    }

    results = {}
    for cls_idx, cls in enumerate(classes):
        sim = sim_matrix[:, cls_idx]
        labels = labels_by_class[cls]
        results[cls] = {}

        for k in k_list:
            recall, precision, ndcg = evaluate_class(sim, labels, k)
            results[cls][k] = {
                "recall": recall,
                "precision": precision,
                "ndcg": ndcg,
            }

    # Macro average across classes
    macro = {k: {} for k in k_list}
    for k in k_list:
        for metric in ["recall", "precision", "ndcg"]:
            macro[k][metric] = float(np.mean([results[cls][k][metric] for cls in classes]))

    return {
        "model_key": model_key,
        "model_label": model_label,
        "results_by_class": results,
        "macro": macro,
    }


def evaluate_all_models(model_keys: list, dataset: list, classes: list, k_list: list, limit: int):
    dataset_subset = dataset[:limit]
    print(f"Evaluation subset size: {len(dataset_subset)}")

    all_results = {}
    for mk in model_keys:
        all_results[mk] = evaluate_single_model(mk, dataset_subset, classes, k_list)
        clear_model_cache()

    return all_results, dataset_subset


MODEL_KEYS = ["align", "clip", "blip", "siglip"]
all_results, dataset_subset = evaluate_all_models(
    model_keys=MODEL_KEYS,
    dataset=dataset,
    classes=CLASSES,
    k_list=K_LIST,
    limit=LIMIT,
)

# Quick macro-metrics print for sanity check
for mk in MODEL_KEYS:
    label = all_results[mk]["model_label"]
    print()
    print(f"{label} macro metrics:")
    for k in K_LIST:
        m = all_results[mk]["macro"][k]
        print(f"K={k:>2} | Recall={m['recall']:.4f} | Precision={m['precision']:.4f} | nDCG={m['ndcg']:.4f}")


Evaluation subset size: 2000

===== Evaluating ALIGN =====
Loading ALIGN from kakaobrain/align-base...


Images [ALIGN]:   0%|          | 0/125 [00:00<?, ?it/s]

## 4) Visualization: Metrics Side-by-Side per Model


In [ ]:
SELECTED_CLASSES = [
    "car",
    "truck",
    "motorcycle",
    "bicycle",
    "pedestrian",
    "barrier",
    "trafficcone",
]


def plot_metric_side_by_side(all_results: dict, model_keys: list, metric: str, selected_classes: list):
    fig, axes = plt.subplots(1, len(model_keys), figsize=(5 * len(model_keys), 4), sharey=True)

    if len(model_keys) == 1:
        axes = [axes]

    for ax, mk in zip(axes, model_keys):
        payload = all_results[mk]
        label = payload["model_label"]
        res = payload["results_by_class"]

        for cls in selected_classes:
            if cls not in res:
                continue
            ks = sorted(res[cls].keys())
            vals = [res[cls][k][metric] for k in ks]
            ax.plot(ks, vals, marker="o", linewidth=1.8, label=cls)

        ax.set_title(label)
        ax.set_xlabel("K")
        ax.grid(True, alpha=0.35)

    axes[0].set_ylabel(metric.upper())
    axes[-1].legend(loc="best", fontsize=8)
    fig.suptitle(f"{metric.upper()}@K by model", y=1.03)
    plt.tight_layout()
    plt.show()


def plot_macro_comparison(all_results: dict, model_keys: list, k_list: list):
    metrics = ["recall", "precision", "ndcg"]
    fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharex=True)

    for ax, metric in zip(axes, metrics):
        for mk in model_keys:
            label = all_results[mk]["model_label"]
            vals = [all_results[mk]["macro"][k][metric] for k in k_list]
            ax.plot(k_list, vals, marker="o", linewidth=2, label=label)

        ax.set_title(f"Macro {metric.upper()}@K")
        ax.set_xlabel("K")
        ax.grid(True, alpha=0.35)

    axes[0].set_ylabel("score")
    axes[-1].legend(loc="best")
    plt.tight_layout()
    plt.show()


plot_metric_side_by_side(all_results, MODEL_KEYS, metric="recall", selected_classes=SELECTED_CLASSES)
plot_metric_side_by_side(all_results, MODEL_KEYS, metric="precision", selected_classes=SELECTED_CLASSES)
plot_metric_side_by_side(all_results, MODEL_KEYS, metric="ndcg", selected_classes=SELECTED_CLASSES)

plot_macro_comparison(all_results, MODEL_KEYS, K_LIST)


## 5) Inference Speed Benchmark

Measure average inference speed separately for images and text.


In [ ]:
def benchmark_model_speed(
    model_key: str,
    dataset_subset: list,
    text_prompts: list,
    image_batch_size: int = IMAGE_BATCH_SIZE,
    text_batch_size: int = TEXT_BATCH_SIZE,
    n_images: int = 256,
    n_texts: int = 512,
):
    label = MODEL_SPECS[model_key]["label"]

    image_paths = [x["image_path"] for x in dataset_subset[:n_images]]
    texts = text_prompts[:n_texts]

    # Warmup
    _ = encode_images_from_paths(model_key, image_paths[:min(image_batch_size, len(image_paths))], batch_size=image_batch_size)
    _ = encode_texts(model_key, texts[:min(text_batch_size, len(texts))], batch_size=text_batch_size)

    # Image timing
    t0 = time.perf_counter()
    _ = encode_images_from_paths(model_key, image_paths, batch_size=image_batch_size)
    image_time = time.perf_counter() - t0

    # Text timing
    t1 = time.perf_counter()
    _ = encode_texts(model_key, texts, batch_size=text_batch_size)
    text_time = time.perf_counter() - t1

    image_ips = len(image_paths) / image_time if image_time > 0 else 0.0
    text_tps = len(texts) / text_time if text_time > 0 else 0.0

    return {
        "model_key": model_key,
        "model_label": label,
        "n_images": len(image_paths),
        "n_texts": len(texts),
        "image_sec_total": float(image_time),
        "text_sec_total": float(text_time),
        "image_items_per_sec": float(image_ips),
        "text_items_per_sec": float(text_tps),
        "image_ms_per_item": float((image_time / len(image_paths)) * 1000),
        "text_ms_per_item": float((text_time / len(texts)) * 1000),
    }


def build_text_prompts(classes: list, n_repeat: int = 16) -> list:
    # Increase number of texts for more stable timing
    base = [f"photo of a {c}" for c in classes]
    return (base * n_repeat)


speed_rows = []
bench_texts = build_text_prompts(CLASSES, n_repeat=16)

for mk in MODEL_KEYS:
    print()
    print(f"Benchmarking {MODEL_SPECS[mk]['label']}...")
    row = benchmark_model_speed(
        model_key=mk,
        dataset_subset=dataset_subset,
        text_prompts=bench_texts,
        n_images=min(256, len(dataset_subset)),
        n_texts=min(512, len(bench_texts)),
    )
    speed_rows.append(row)
    print(
        f"{row['model_label']}: "
        f"image={row['image_items_per_sec']:.2f} img/s ({row['image_ms_per_item']:.2f} ms/img), "
        f"text={row['text_items_per_sec']:.2f} txt/s ({row['text_ms_per_item']:.2f} ms/txt)"
    )
    clear_model_cache()


# Plot: inference throughput (items/sec)
labels = [r["model_label"] for r in speed_rows]
img_speed = [r["image_items_per_sec"] for r in speed_rows]
txt_speed = [r["text_items_per_sec"] for r in speed_rows]

x = np.arange(len(labels))
width = 0.35

plt.figure(figsize=(10, 5))
plt.bar(x - width / 2, img_speed, width, label="Image items/sec")
plt.bar(x + width / 2, txt_speed, width, label="Text items/sec")
plt.xticks(x, labels)
plt.ylabel("items/sec")
plt.title("Inference speed by model")
plt.grid(axis="y", alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


# Plot: latency (ms/item)
img_latency = [r["image_ms_per_item"] for r in speed_rows]
txt_latency = [r["text_ms_per_item"] for r in speed_rows]

plt.figure(figsize=(10, 5))
plt.bar(x - width / 2, img_latency, width, label="Image ms/item")
plt.bar(x + width / 2, txt_latency, width, label="Text ms/item")
plt.xticks(x, labels)
plt.ylabel("ms/item")
plt.title("Inference latency by model")
plt.grid(axis="y", alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()
